# Feature Exploration: Route Distance (Multi-Route)

This notebook explores the addition of route distance as a feature to the multi-route model. The rationale is as follows:
- Longer routes may have different delay characteristics (may be less sensitive to individual delays)
- Distance affects turnaround time, fuel requirements, crew scheduling
- Could interact with weather (longer exposure)

Note that the baseline model used here has holiday features (based on the previous notebook added [7a](/notebooks/7a_feature_exploration_holidays_multiroute.ipynb)).

**New Feature to Test:**
- `route_distance_km`: Distance between airports
- Also test normalised version: `route_distance_norm`


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import date
from calendar import monthrange
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, f1_score, roc_auc_score
)
import warnings
warnings.filterwarnings('ignore')

# For holiday generation
import holidays

# Plotting style
plt.rcParams['text.usetex'] = False
plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.linewidth'] = 1.5

# XGBoost for classification
try:
    import xgboost as xgb
    HAS_XGB = True
    print("XGBoost available")
except ImportError:
    HAS_XGB = False
    print("XGBoost not installed")

%matplotlib inline

## 1. Define Route Distances

Great circle distances between Australian airports (in km):
- Sydney (SYD): -33.9461, 151.1772
- Melbourne (MEL): -37.6690, 144.8410
- Hobart (HBA): -42.8361, 147.5102

In [ ]:
# Airport coordinates (latitude, longitude)
AIRPORT_COORDS = {
    'Sydney': (-33.9461, 151.1772),
    'Melbourne': (-37.6690, 144.8410),
    'Hobart': (-42.8361, 147.5102)
}

def haversine_distance(coord1, coord2):
    """
    Calculate the great circle distance between two points 
    on the earth (specified in decimal degrees).
    Returns distance in kilometers.
    """
    from math import radians, cos, sin, asin, sqrt
    
    lat1, lon1 = coord1
    lat2, lon2 = coord2
    
    # Convert to radians
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    
    # Haversine formula
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a))
    
    # Earth's radius in kilometers
    r = 6371
    
    return c * r

# Calculate distances for all route pairs
ROUTE_DISTANCES = {}
cities = list(AIRPORT_COORDS.keys())

for city1 in cities:
    for city2 in cities:
        if city1 != city2:
            route = f"{city1}_{city2}"
            dist = haversine_distance(AIRPORT_COORDS[city1], AIRPORT_COORDS[city2])
            ROUTE_DISTANCES[route] = round(dist, 1)

print("Route Distances (km):")
print("=" * 40)
for route, dist in sorted(ROUTE_DISTANCES.items()):
    print(f"  {route:<25} {dist:>8.1f} km")

## 2. Data Preparation

In [ ]:
# Load multi-route data
df = pd.read_csv('../data/processed/ml_training_data_multiroute.csv')

# Parse dates
df['year_month_dt'] = pd.to_datetime(df['year_month'])
df['month_num'] = df['year_month_dt'].dt.month
df['year'] = df['year'].astype(int)

# Create unique identifier for each airline-route combination
df['airline_route'] = df['airline'] + '_' + df['departing_port'] + '_' + df['arriving_port']
df['route'] = df['departing_port'] + '_' + df['arriving_port']

# Sort for proper lag creation
df = df.sort_values(['airline_route', 'year_month_dt']).reset_index(drop=True)

print(f"Original shape: {df.shape}")
print(f"Date range: {df['year_month'].min()} to {df['year_month'].max()}")

In [ ]:
# Add route distance feature
df['route_distance_km'] = df['route'].map(ROUTE_DISTANCES)

# Check mapping worked
print("Route distance mapping:")
print(df.groupby('route')['route_distance_km'].first())

# Create normalized version (0-1 scale)
df['route_distance_norm'] = df['route_distance_km'] / df['route_distance_km'].max()

print(f"\nDistance range: {df['route_distance_km'].min():.1f} - {df['route_distance_km'].max():.1f} km")

## 3. Generate Holiday Features (from 7a)

In [ ]:
# Create holiday calendars
years = list(range(2010, 2026))
STATES = ['ACT', 'NSW', 'NT', 'QLD', 'SA', 'TAS', 'VIC', 'WA']

state_holidays = {}
for state in STATES:
    state_holidays[state] = holidays.Australia(years=years, prov=state)

national_holidays = holidays.Australia(years=years)

# Helper functions
def count_holidays_in_month(holiday_dict, year, month):
    count = 0
    _, days_in_month = monthrange(year, month)
    for day in range(1, days_in_month + 1):
        if date(year, month, day) in holiday_dict:
            count += 1
    return count

def has_major_holiday_in_month(holiday_dict, year, month):
    major_keywords = ['christmas', 'easter', 'anzac', 'good friday', 'boxing']
    _, days_in_month = monthrange(year, month)
    for day in range(1, days_in_month + 1):
        d = date(year, month, day)
        if d in holiday_dict:
            name = holiday_dict[d].lower()
            if any(kw in name for kw in major_keywords):
                return 1
    return 0

def get_school_holiday_periods(year):
    return [
        (date(year-1, 12, 18), date(year, 1, 28)),
        (date(year, 4, 8), date(year, 4, 23)),
        (date(year, 6, 27), date(year, 7, 14)),
        (date(year, 9, 23), date(year, 10, 7)),
        (date(year, 12, 18), date(year, 12, 31)),
    ]

def count_school_holiday_days_in_month(year, month):
    periods = get_school_holiday_periods(year)
    if month == 1:
        periods.extend(get_school_holiday_periods(year))
    _, days_in_month = monthrange(year, month)
    holiday_days = 0
    for day in range(1, days_in_month + 1):
        current_date = date(year, month, day)
        for start, end in periods:
            if start <= current_date <= end:
                holiday_days += 1
                break
    return holiday_days

print("Holiday functions defined.")

In [ ]:
# Generate holiday features
year_months = df[['year', 'month_num']].drop_duplicates().sort_values(['year', 'month_num'])
print(f"Generating holiday features for {len(year_months)} unique months...")

holiday_features = []
for _, row in year_months.iterrows():
    year, month = int(row['year']), int(row['month_num'])
    _, days_in_month = monthrange(year, month)
    
    feature_row = {'year': year, 'month_num': month}
    
    # National and total holidays
    feature_row['n_public_holidays_national'] = count_holidays_in_month(national_holidays, year, month)
    
    all_holiday_dates = set()
    for state in STATES:
        for day in range(1, days_in_month + 1):
            d = date(year, month, day)
            if d in state_holidays[state]:
                all_holiday_dates.add(d)
    feature_row['n_public_holidays_total'] = len(all_holiday_dates)
    
    # Major holiday flag
    has_major = 0
    for state in STATES:
        if has_major_holiday_in_month(state_holidays[state], year, month):
            has_major = 1
            break
    feature_row['has_major_holiday'] = has_major
    
    # School holidays
    school_days = count_school_holiday_days_in_month(year, month)
    feature_row['school_holiday_days'] = school_days
    feature_row['pct_school_holiday'] = round(school_days / days_in_month, 4)
    
    holiday_features.append(feature_row)

df_holidays = pd.DataFrame(holiday_features)
df = df.merge(df_holidays, on=['year', 'month_num'], how='left')
print(f"Shape after adding holidays: {df.shape}")

## 4. Filter Low-Volume Airline-Routes

In [ ]:
volume_threshold = 50

airline_route_volume = df.groupby('airline_route')['sectors_scheduled'].mean().reset_index()
airline_route_volume.columns = ['airline_route', 'avg_volume']
high_volume_ar = airline_route_volume[airline_route_volume['avg_volume'] >= volume_threshold]['airline_route'].tolist()

df_filtered = df[df['airline_route'].isin(high_volume_ar)].copy()

print(f"Volume threshold: {volume_threshold} flights/month")
print(f"Records before filtering: {len(df)}")
print(f"Records after filtering:  {len(df_filtered)}")
print(f"Remaining airline-routes: {df_filtered['airline_route'].nunique()}")

## 5. Explore Distance Feature

In [ ]:
# Correlation with delay_rate
print("Distance Feature Correlations:")
print("=" * 50)
print(f"  route_distance_km:   {df_filtered['route_distance_km'].corr(df_filtered['delay_rate']):.4f}")
print(f"  route_distance_norm: {df_filtered['route_distance_norm'].corr(df_filtered['delay_rate']):.4f}")

# By route
print("\nDelay Rate Statistics by Route Distance:")
print("-" * 60)
route_stats = df_filtered.groupby('route').agg({
    'route_distance_km': 'first',
    'delay_rate': ['mean', 'std', 'count']
}).round(4)
route_stats.columns = ['distance_km', 'mean_delay', 'std_delay', 'n_obs']
route_stats = route_stats.sort_values('distance_km')
print(route_stats)

In [ ]:
# Visualize distance vs delay
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Plot 1: Boxplot of delay rate by route (ordered by distance)
ax = axes[0]
route_order = route_stats.index.tolist()
delay_by_route = [df_filtered[df_filtered['route'] == r]['delay_rate'].values for r in route_order]
bp = ax.boxplot(delay_by_route, labels=[f"{r.replace('_', '→')}\n({int(route_stats.loc[r, 'distance_km'])} km)" for r in route_order])
ax.set_ylabel('Delay Rate')
ax.set_title('Delay Rate by Route (ordered by distance)')
ax.tick_params(axis='x', rotation=45)
ax.grid(True, alpha=0.3, axis='y')

# Plot 2: Scatter of distance vs mean delay
ax = axes[1]
ax.scatter(route_stats['distance_km'], route_stats['mean_delay'], s=100, alpha=0.8)
for route in route_stats.index:
    ax.annotate(route.replace('_', '→'), 
                (route_stats.loc[route, 'distance_km'], route_stats.loc[route, 'mean_delay']),
                fontsize=8, ha='center', va='bottom')
ax.set_xlabel('Route Distance (km)')
ax.set_ylabel('Mean Delay Rate')
ax.set_title('Mean Delay Rate vs Route Distance')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Feature Engineering

In [ ]:
# Create lagged features
df_filtered['delay_rate_lag1'] = df_filtered.groupby('airline_route')['delay_rate'].shift(1)
df_filtered['delay_rate_lag2'] = df_filtered.groupby('airline_route')['delay_rate'].shift(2)
df_filtered['delay_rate_gradient'] = df_filtered['delay_rate_lag1'] - df_filtered['delay_rate_lag2']

# Cyclical month encoding
df_filtered['month_sin'] = np.sin(2 * np.pi * df_filtered['month_num'] / 12)
df_filtered['month_cos'] = np.cos(2 * np.pi * df_filtered['month_num'] / 12)

# One-hot encode airline and route
airline_dummies = pd.get_dummies(df_filtered['airline'], prefix='airline')
df_filtered = pd.concat([df_filtered, airline_dummies], axis=1)
airline_cols = list(airline_dummies.columns)

route_dummies = pd.get_dummies(df_filtered['route'], prefix='route')
df_filtered = pd.concat([df_filtered, route_dummies], axis=1)
route_cols = list(route_dummies.columns)

# Weather features
df_filtered['rainy_days_arr_exp'] = np.exp(df_filtered['rainy_days_arr'] / df_filtered['rainy_days_arr'].max())
df_filtered['temp_volatility_total'] = df_filtered['temp_volatility_dep'] + df_filtered['temp_volatility_arr']
df_filtered['temp_volatility_total_exp'] = np.exp(df_filtered['temp_volatility_total'] / df_filtered['temp_volatility_total'].max())

# Drop rows with missing lag values
df_clean = df_filtered.dropna(subset=['delay_rate_lag1', 'delay_rate_lag2', 'delay_rate_gradient']).copy()
print(f"Rows after dropping NaN: {len(df_clean)}")

## 7. Train/Validation/Test Split

In [ ]:
train_mask = ((df_clean['year'] <= 2017) | (df_clean['year'] == 2024))
val_mask = ((df_clean['year'] == 2018) | (df_clean['year'] == 2023))
test_mask = ((df_clean['year'] == 2019) | (df_clean['year'] >= 2025))

print(f"Train (2010-2017, 2024): {train_mask.sum()} samples")
print(f"Validation (2018, 2023): {val_mask.sum()} samples")
print(f"Test (2019, 2025):       {test_mask.sum()} samples")

In [ ]:
# Define feature sets
base_features = airline_cols + route_cols + ['month_sin', 'month_cos', 'delay_rate_lag1', 'sectors_scheduled']

# Weather features
weather_features = ['rainy_days_arr_exp', 'delay_rate_gradient', 'temp_volatility_total_exp']

# Holiday features (all holidays from 7a)
holiday_features = ['n_public_holidays_national', 'n_public_holidays_total', 'has_major_holiday',
                    'school_holiday_days', 'pct_school_holiday']

# Baseline = 7a with all holidays
baseline_features = base_features + weather_features + holiday_features

# Feature variants to test
distance_variants = {
    'baseline (7a)': baseline_features,
    '+distance_km': baseline_features + ['route_distance_km'],
    '+distance_norm': baseline_features + ['route_distance_norm'],
}

print("Feature variants:")
for name, features in distance_variants.items():
    print(f"  {name}: {len(features)} features")

In [ ]:
# Prepare target variables
y_train_reg = df_clean.loc[train_mask, 'delay_rate'].values
y_val_reg = df_clean.loc[val_mask, 'delay_rate'].values
y_test_reg = df_clean.loc[test_mask, 'delay_rate'].values

y_train_clf = df_clean.loc[train_mask, 'is_high_delay'].values
y_val_clf = df_clean.loc[val_mask, 'is_high_delay'].values
y_test_clf = df_clean.loc[test_mask, 'is_high_delay'].values

print("Target variables prepared.")

## 8. Regression Models

In [ ]:
reg_results = []

for variant_name, features in distance_variants.items():
    print(f"\n{'='*60}")
    print(f"Regression with: {variant_name}")
    print(f"{'='*60}")
    
    X_train = df_clean.loc[train_mask, features].values
    X_val = df_clean.loc[val_mask, features].values
    X_test = df_clean.loc[test_mask, features].values
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)
    
    # Ridge
    ridge = Ridge(alpha=1.0)
    ridge.fit(X_train_scaled, y_train_reg)
    ridge_test_pred = ridge.predict(X_test_scaled)
    ridge_r2 = r2_score(y_test_reg, ridge_test_pred)
    ridge_rmse = np.sqrt(mean_squared_error(y_test_reg, ridge_test_pred))
    print(f"  Ridge:    R²={ridge_r2:.4f}, RMSE={ridge_rmse:.4f}")
    reg_results.append({'Variant': variant_name, 'Model': 'Ridge', 'Test_R2': ridge_r2, 'Test_RMSE': ridge_rmse})
    
    # Random Forest
    rf_reg = RandomForestRegressor(n_estimators=100, max_depth=10, min_samples_leaf=5, random_state=42, n_jobs=-1)
    rf_reg.fit(X_train, y_train_reg)
    rf_test_pred = rf_reg.predict(X_test)
    rf_r2 = r2_score(y_test_reg, rf_test_pred)
    rf_rmse = np.sqrt(mean_squared_error(y_test_reg, rf_test_pred))
    print(f"  RF:       R²={rf_r2:.4f}, RMSE={rf_rmse:.4f}")
    reg_results.append({'Variant': variant_name, 'Model': 'Random Forest', 'Test_R2': rf_r2, 'Test_RMSE': rf_rmse})

reg_df = pd.DataFrame(reg_results)

## 9. Classification Models

In [ ]:
clf_results = []

for variant_name, features in distance_variants.items():
    print(f"\n{'='*60}")
    print(f"Classification with: {variant_name}")
    print(f"{'='*60}")
    
    X_train = df_clean.loc[train_mask, features].values
    X_val = df_clean.loc[val_mask, features].values
    X_test = df_clean.loc[test_mask, features].values
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)
    
    # Logistic
    logreg = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
    logreg.fit(X_train_scaled, y_train_clf)
    logreg_test_pred = logreg.predict(X_test_scaled)
    logreg_test_proba = logreg.predict_proba(X_test_scaled)[:, 1]
    logreg_f1 = f1_score(y_test_clf, logreg_test_pred)
    logreg_auc = roc_auc_score(y_test_clf, logreg_test_proba)
    print(f"  Logistic: F1={logreg_f1:.4f}, AUC={logreg_auc:.4f}")
    clf_results.append({'Variant': variant_name, 'Model': 'Logistic', 'Test_F1': logreg_f1, 'Test_AUC': logreg_auc})
    
    # Random Forest
    rf_clf = RandomForestClassifier(n_estimators=100, max_depth=10, min_samples_leaf=5, random_state=42, n_jobs=-1)
    rf_clf.fit(X_train, y_train_clf)
    rf_clf_test_pred = rf_clf.predict(X_test)
    rf_clf_test_proba = rf_clf.predict_proba(X_test)[:, 1]
    rf_clf_f1 = f1_score(y_test_clf, rf_clf_test_pred)
    rf_clf_auc = roc_auc_score(y_test_clf, rf_clf_test_proba)
    print(f"  RF Clf:   F1={rf_clf_f1:.4f}, AUC={rf_clf_auc:.4f}")
    clf_results.append({'Variant': variant_name, 'Model': 'Random Forest', 'Test_F1': rf_clf_f1, 'Test_AUC': rf_clf_auc})
    
    # XGBoost
    if HAS_XGB:
        xgb_clf = xgb.XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, min_child_weight=5, random_state=42, n_jobs=-1)
        xgb_clf.fit(X_train, y_train_clf, eval_set=[(X_val, y_val_clf)], verbose=False)
        xgb_test_pred = xgb_clf.predict(X_test)
        xgb_test_proba = xgb_clf.predict_proba(X_test)[:, 1]
        xgb_f1 = f1_score(y_test_clf, xgb_test_pred)
        xgb_auc = roc_auc_score(y_test_clf, xgb_test_proba)
        print(f"  XGBoost:  F1={xgb_f1:.4f}, AUC={xgb_auc:.4f}")
        clf_results.append({'Variant': variant_name, 'Model': 'XGBoost', 'Test_F1': xgb_f1, 'Test_AUC': xgb_auc})

clf_df = pd.DataFrame(clf_results)

## 10. Results Comparison

In [ ]:
# Regression comparison
print("=" * 80)
print("REGRESSION: Baseline (7a) vs +Distance")
print("=" * 80)

reg_pivot = reg_df.pivot(index='Model', columns='Variant', values='Test_R2')
reg_pivot = reg_pivot[['baseline (7a)', '+distance_km', '+distance_norm']]

print(f"\n{'Model':<15} {'baseline':>12} {'+km':>12} {'+norm':>12}")
print("-" * 55)
for model in reg_pivot.index:
    row = reg_pivot.loc[model]
    print(f"{model:<15} {row['baseline (7a)']:>12.4f} {row['+distance_km']:>12.4f} {row['+distance_norm']:>12.4f}")

In [ ]:
# Classification comparison
print("=" * 80)
print("CLASSIFICATION: Baseline (7a) vs +Distance")
print("=" * 80)

clf_pivot = clf_df.pivot(index='Model', columns='Variant', values='Test_F1')
clf_pivot = clf_pivot[['baseline (7a)', '+distance_km', '+distance_norm']]

print(f"\n{'Model':<15} {'baseline':>12} {'+km':>12} {'+norm':>12}")
print("-" * 55)
for model in clf_pivot.index:
    row = clf_pivot.loc[model]
    print(f"{model:<15} {row['baseline (7a)']:>12.4f} {row['+distance_km']:>12.4f} {row['+distance_norm']:>12.4f}")

In [ ]:
# Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Regression
ax = axes[0]
x_reg = np.arange(len(reg_pivot.index))
width = 0.25
colors = ['steelblue', '#27ae60', '#e67e22']

for i, variant in enumerate(['baseline (7a)', '+distance_km', '+distance_norm']):
    ax.bar(x_reg + i*width, reg_pivot[variant], width, label=variant, color=colors[i], alpha=0.8)

ax.set_ylabel(r'Test $R^2$')
ax.set_title(r'Regression: $R^2$ Comparison')
ax.set_xticks(x_reg + width)
ax.set_xticklabels(reg_pivot.index)
ax.legend(fontsize=9)
ax.set_ylim(0, 0.6)
ax.grid(True, alpha=0.3, axis='y')

# Classification
ax = axes[1]
x_clf = np.arange(len(clf_pivot.index))

for i, variant in enumerate(['baseline (7a)', '+distance_km', '+distance_norm']):
    ax.bar(x_clf + i*width, clf_pivot[variant], width, label=variant, color=colors[i], alpha=0.8)

ax.set_ylabel(r'Test $F_1$')
ax.set_title(r'Classification: $F_1$ Comparison')
ax.set_xticks(x_clf + width)
ax.set_xticklabels(clf_pivot.index)
ax.legend(fontsize=9)
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 11. Feature Importance

In [ ]:
# Train RF with distance_km to check feature importance
features_with_dist = baseline_features + ['route_distance_km']
X_train_reg = df_clean.loc[train_mask, features_with_dist].values

rf_final = RandomForestRegressor(n_estimators=100, max_depth=10, min_samples_leaf=5, random_state=42, n_jobs=-1)
rf_final.fit(X_train_reg, y_train_reg)

importance_df = pd.DataFrame({
    'Feature': features_with_dist,
    'Importance': rf_final.feature_importances_
}).sort_values('Importance', ascending=False)

print("Feature Importance (RF Regression with distance):")
print("-" * 60)
for _, row in importance_df.head(15).iterrows():
    marker = " <- DISTANCE" if 'distance' in row['Feature'] else ""
    print(f"  {row['Feature']:<35} {row['Importance']:.4f}{marker}")

# Find distance rank
dist_rank = list(importance_df['Feature']).index('route_distance_km') + 1
dist_importance = importance_df[importance_df['Feature'] == 'route_distance_km']['Importance'].values[0]
print(f"\nroute_distance_km: rank {dist_rank}, importance {dist_importance:.4f}")

## 12. Summary and Observations

In [ ]:
print("="*80)
print("SUMMARY: Impact of Route Distance on Multi-Route Model")
print("="*80)

# Summary metrics table - Regression
print("\nREGRESSION PERFORMANCE:")
print("-" * 70)
print(f"{'Model':<15} {'7a Baseline':>12} {'7b +Dist':>12} {'Δ R²':>10} {'Status':>12}")
print("-" * 70)

for model in reg_pivot.index:
    baseline = reg_pivot.loc[model, 'baseline (7a)']
    best = max(reg_pivot.loc[model, '+distance_km'], reg_pivot.loc[model, '+distance_norm'])
    diff = best - baseline
    sign = '+' if diff > 0 else ''
    status = 'IMPROVED' if diff > 0.005 else ('WORSE' if diff < -0.005 else '~same')
    print(f"{model:<15} {baseline:>12.4f} {best:>12.4f} {sign}{diff:>9.4f} {status:>12}")

# Summary metrics table - Classification
print("\nCLASSIFICATION PERFORMANCE:")
print("-" * 70)
print(f"{'Model':<15} {'7a Baseline':>12} {'7b +Dist':>12} {'Δ F1':>10} {'Status':>12}")
print("-" * 70)

for model in clf_pivot.index:
    baseline = clf_pivot.loc[model, 'baseline (7a)']
    best = max(clf_pivot.loc[model, '+distance_km'], clf_pivot.loc[model, '+distance_norm'])
    diff = best - baseline
    sign = '+' if diff > 0 else ''
    status = 'IMPROVED' if diff > 0.005 else ('WORSE' if diff < -0.005 else '~same')
    print(f"{model:<15} {baseline:>12.4f} {best:>12.4f} {sign}{diff:>9.4f} {status:>12}")

### Observations:
- No model shows improvement
- Low feature importance: 17 out of 26
- With the scope being only the major Australian domestic routes (low number of routes), perhaps there is not sufficient variation to capture the effects of route distance


## 13. Next Step

Route distance will not be added as a feature. Next, the impact of adding another weather indicator, number of extreme weather days, will be investigated.

See notebook [7c_feature_exploration_extreme_weather.ipynb](/notebooks/7c_feature_exploration_extreme_weather.ipynb).